# בדיקת מודל Torah RAV

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "unsloth/Qwen3.5-4B"
adapter_path = r"c:\Users\rubin\Documents\Chavruta.AI\data\torah_rav_final"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()
print("מודל נטען בהצלחה")

In [ ]:
def ask(question, max_new_tokens=400, temperature=0.7):
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"שאלה: {question}")
    print("-" * 60)
    print(response)

## שאלות בדיקה — שנה את השאלה לכל מה שרוצה

In [ ]:
# שאלה על פרשנות רש"י
ask("מה פירוש רש\"י על 'בראשית ברא אלהים'?")

In [ ]:
# שאלה על רמב"ן
ask("מה הוסיף הרמב\"ן על פירוש רש\"י בפרשת בראשית?")

In [ ]:
# שאלה על מחלוקת רש"י ורמב"ן
ask("מה ההבדל בין פירוש רש\"י לפירוש הרמב\"ן על סיפור הבריאה?")